In [ ]:
import yaml
import os


import numpy as np
from src.estate import RentingScenario, BuyingScenario, compare_scenarios
from src.tax import NLHomeTax2026

In [ ]:
def load_scenario_config(file_path: str) -> dict:
    with open(file_path, "r") as file:
        return yaml.safe_load(file)

def list_files_in_directory(directory_path):
    try:
        files = [f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]
        return files
    except FileNotFoundError:
        print(f"The directory {directory_path} does not exist.")
        return []

# Example usage
directory_path = "scenarios"
files = sorted(list_files_in_directory(directory_path))
files = [fn for fn in files if fn.endswith('.yaml')]
files

In [ ]:
ind = -3
file = files[ind]
file

In [ ]:
# Load the scenario configuration
scenario_file = os.path.join("scenarios", file)
config = load_scenario_config(scenario_file)

# Access the configuration
print(config)

In [ ]:
config["renting"]

In [ ]:
rent = RentingScenario(**config["renting"])
rent

In [ ]:
tax = NLHomeTax2026(**config["tax"])
tax

In [ ]:
buy = BuyingScenario(**config["buying"], tax=tax)
buy

In [ ]:
results = compare_scenarios(rent, buy)
wealth_r = results["renting"]["wealth_end"] + results["invest"]["renting_invests"]
wealth_b = results["buying"]["wealth_end"] + results["invest"]["buying_invests"]
for k, v in results.items():
    print("\n===", k.upper(), "===")

    for kk in v.keys():
        if kk in "monthly_net_cost":
            continue
        val_str = f"{v[kk]:,.2f}" if isinstance(v[kk], (float, int)) else str(v[kk])
        print(f"{kk:>30}: {val_str}")

print()
print(f"Wealth difference: {wealth_b - wealth_r:,.0f}€ (pos = buying saved money)")

In [ ]:
np.array(
    [
        results["renting"]["monthly_net_cost"],
        results["buying"]["monthly_net_cost"]
    ]
).T

In [ ]:
invest_res = results["invest"]
investment_profit_r = max(0, invest_res["renting_invests"] - invest_res["principal_r"])
investment_profit_b = max(0, invest_res["buying_invests"] - invest_res["principal_r"])
print(f"Investment profits:\n\trenting: {investment_profit_r:,.2f} €\n\tbuying: {investment_profit_b:,.2f} €")

In [ ]:
print(f"For scenario '{config['name']}' from file {file}")
print(f"Rental bottom line: {wealth_r:,.2f}")
print(f"Buying bottom line: {wealth_b:,.2f}")
print(f"Buying is better than rental by: {wealth_b - wealth_r:,.2f}")